w = np.random.rand(v,D)

c = np.random.rand(v,D)


get positive samples

get negative samples

generate likelihood

generate gradient

update c,w

In [ ]:
import numpy as np
import random

In [ ]:
text = open("/content/smalltext8.txt").read()

In [ ]:
from collections import Counter

In [ ]:
text = text.split()
print(text[0:10])

vocab = list(set(text))
word2idx = {w: idx for (idx, w) in enumerate(vocab)}
idx2word = {idx: w for (w, idx) in word2idx.items()}

V = len(vocab)

text = [word2idx[w] for w in text]

['\ufeffanarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against']


In [ ]:
dim = 100
window = 2
neg_samples = 5
lr = 0.025
epochs = 1

In [ ]:
freq = Counter(text)
p = np.array([freq[idx] for idx in range(V)])
p = p**(0.75)
p = p/np.sum(p)

print(len(p))
print(V)


467
467


rand -> random values form uniform ditribution (0,1)

randn -> random values from gaussian distribution

In [ ]:
W = np.random.randn(V,dim)
C = np.random.randn(V,dim)

dL/dCi = -(1-sigma)*w

dL/dCj = sigma*w

dL/dw = -(1-sigma)*Ci + {summation(1...k) [sigma*Cj]}

positive samples = 1
negative samples = 5

In [ ]:
def sigmoid(z):
  return 1/(1+np.exp(-z))

In [ ]:
for _ in range(epochs):
  for i, current_word_idx in enumerate(text):
    context_word_indices_for_current_word = []
    for j in range(-window, window + 1):
      if j == 0:
        continue
      context_position = i + j
      if 0 <= context_position < len(text):
        context_word_indices_for_current_word.append(text[context_position])

    forbidden = set(context_word_indices_for_current_word)
    forbidden.add(current_word_idx)

    for context_word_idx in context_word_indices_for_current_word:
      score = sigmoid(np.dot(W[current_word_idx], C[context_word_idx]))
      scale = lr * (score - 1)

      W[current_word_idx] = W[current_word_idx] - (scale) * C[context_word_idx]
      C[context_word_idx] = C[context_word_idx] - (scale) * W[current_word_idx]

    neg_samples_list = []
    while len(neg_samples_list) < neg_samples:
      negative_word_idx = np.random.choice(V, p=p)
      if negative_word_idx not in forbidden:
        neg_samples_list.append(negative_word_idx)

    for negative_word_idx in neg_samples_list:
      score = sigmoid(np.dot(-W[current_word_idx], C[negative_word_idx]))
      scale = lr * score

      W[current_word_idx] = W[current_word_idx] - (scale) * C[negative_word_idx]
      C[negative_word_idx] = C[negative_word_idx] - (scale) * W[current_word_idx]

In [ ]:
Emb = W

def similar(word,top = 5):
  if word not in word2idx:
    print(f"Word '{word}' not in vocabulary.")
    return []
  v = Emb[word2idx[word]]

  #cosine similarity: (A . B) / (||A|| * ||B||)
  dot_products = np.dot(Emb, v)
  norm_emb = np.linalg.norm(Emb, axis=1)
  norm_v = np.linalg.norm(v)

  epsilon = 1e-8
  similarities = dot_products / (norm_emb * norm_v + epsilon)

  ids = np.argsort(-similarities)[1:top+1]

  return [idx2word[i] for i in ids]

In [ ]:
print(similar('king'))

['was', 'its', 'exploiting', 'rulers', 'sans']
